In [1]:
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu132

# EDA

In [2]:
from seqeval.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    AutoTokenizer
)
from datasets import Dataset
import json
import numpy as np

C:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Reproducibility (tambahan review)

Notebook ini tetap memakai model Kevin, `bert-base-multilingual-cased`. Cell asli tidak diubah. Cell tambahan berikut menetapkan seed sebelum tokenizer, model, dan Trainer dibuat agar proses dapat diulang dengan konfigurasi acak yang sama.


In [3]:
import os
import random
import torch
from transformers import set_seed

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

print(f"Random seed: {SEED}")


Random seed: 42


### Path dataset untuk VS Code

Cell berikut mencari root repository secara otomatis, baik kernel dijalankan dari root workspace maupun dari folder notebook. Working directory kemudian diarahkan ke `src/alamatin` agar path relatif pada cell Kevin tetap valid.


In [4]:
from pathlib import Path


def find_repo_root(start_path):
    """Cari folder repo yang memiliki data synthetic dan src/alamatin."""
    start_path = Path(start_path).resolve()
    for candidate in (start_path, *start_path.parents):
        train_file = candidate / "data" / "synthetic" / "train.json"
        notebook_dir = candidate / "src" / "alamatin"
        if train_file.is_file() and notebook_dir.is_dir():
            return candidate
    raise FileNotFoundError(
        "Root repo tidak ditemukan. Buka folder repo ALAMATIN di VS Code, "
        "lalu restart kernel notebook."
    )


REPO_ROOT = find_repo_root(Path.cwd())
NOTEBOOK_WORKDIR = REPO_ROOT / "src" / "alamatin"
DATA_DIR = REPO_ROOT / "data" / "synthetic"

required_datasets = ["train.json", "val.json", "test.json"]
missing_datasets = [
    filename
    for filename in required_datasets
    if not (DATA_DIR / filename).is_file()
]
if missing_datasets:
    raise FileNotFoundError(
        f"Dataset tidak ditemukan di {DATA_DIR}: {missing_datasets}"
    )

os.chdir(NOTEBOOK_WORKDIR)

print(f"Repo root : {REPO_ROOT}")
print(f"Dataset   : {DATA_DIR}")
print(f"Kernel cwd: {Path.cwd()}")


Repo root : D:\Kuliah\Lomba Comfest AIC
Dataset   : D:\Kuliah\Lomba Comfest AIC\data\synthetic
Kernel cwd: D:\Kuliah\Lomba Comfest AIC\src\alamatin


In [5]:
with open('../../data/synthetic/train.json', 'r') as f:
    data = json.load(f)
with open('../../data/synthetic/val.json', 'r') as f:
    eval_data = json.load(f)

In [6]:
print(type(data))
print(data.keys())

<class 'dict'>
dict_keys(['schema_version', 'generator_version', 'template_version', 'label_order', 'examples'])


In [7]:
data['label_order']

['O',
 'B-JALAN',
 'I-JALAN',
 'B-NOMOR',
 'I-NOMOR',
 'B-RT',
 'I-RT',
 'B-RW',
 'I-RW',
 'B-KELURAHAN',
 'I-KELURAHAN',
 'B-KECAMATAN',
 'I-KECAMATAN',
 'B-KOTA_KABUPATEN',
 'I-KOTA_KABUPATEN',
 'B-PROVINSI',
 'I-PROVINSI',
 'B-KODEPOS',
 'I-KODEPOS',
 'B-DETAIL_LOKASI',
 'I-DETAIL_LOKASI']

In [8]:
print(type(data['examples']))
print(f'data length: {len(data['examples'])}')

<class 'list'>
data length: 4500


In [9]:
print(f'{type(data['examples'][0])}')
print(f'length of 1 data: {len(data['examples'][0])}')
print(f'keys of 1 data: {data['examples'][0].keys()}')

<class 'dict'>
length of 1 data: 4
keys of 1 data: dict_keys(['id', 'categories', 'tokens', 'labels'])


In [10]:
print(data['examples'][0]['tokens'])
print()
print(data['examples'][0]['labels'])

['JL.', 'AHMADY', 'ANI', 'NOMER', '177', 'DESA', 'CIBUNAR', 'TAROGONG', 'KIDUL', ',', 'KAB.', 'GARUT', '44129']

['B-JALAN', 'I-JALAN', 'I-JALAN', 'B-NOMOR', 'I-NOMOR', 'B-KELURAHAN', 'I-KELURAHAN', 'B-KECAMATAN', 'I-KECAMATAN', 'O', 'B-KOTA_KABUPATEN', 'I-KOTA_KABUPATEN', 'B-KODEPOS']


# Preprocessing

In [11]:
examples = data['examples']
eval_examples = eval_data['examples']

In [12]:
label2id = {
    label : i for i,label in enumerate(data['label_order'])
}
id2label = {
    i : label for i,label in enumerate(data['label_order'])
}
label2id

{'O': 0,
 'B-JALAN': 1,
 'I-JALAN': 2,
 'B-NOMOR': 3,
 'I-NOMOR': 4,
 'B-RT': 5,
 'I-RT': 6,
 'B-RW': 7,
 'I-RW': 8,
 'B-KELURAHAN': 9,
 'I-KELURAHAN': 10,
 'B-KECAMATAN': 11,
 'I-KECAMATAN': 12,
 'B-KOTA_KABUPATEN': 13,
 'I-KOTA_KABUPATEN': 14,
 'B-PROVINSI': 15,
 'I-PROVINSI': 16,
 'B-KODEPOS': 17,
 'I-KODEPOS': 18,
 'B-DETAIL_LOKASI': 19,
 'I-DETAIL_LOKASI': 20}

## tokenizing

In [13]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-multilingual-cased')

D:\Kuliah\Lomba Comfest AIC\.venv-run\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ACER\.cache\huggingface\hub\models--bert-base-multilingual-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [14]:
tokens = examples[0]['tokens']

encoding = tokenizer(tokens, is_split_into_words = True)
print(encoding.tokens())

['[CLS]', 'J', '##L', '.', 'AH', '##MA', '##D', '##Y', 'AN', '##I', 'NO', '##ME', '##R', '177', 'DE', '##SA', 'C', '##IB', '##UN', '##AR', 'TA', '##RO', '##GO', '##NG', 'K', '##ID', '##UL', ',', 'KA', '##B', '.', 'GA', '##R', '##UT', '441', '##2', '##9', '[SEP]']


In [15]:
# ngetes pake kalimat for better understanding

# tokens = 'Jl. Betonmas Selatan no. 185'

# encoding = tokenizer(tokens, is_split_into_words = False)
# print(encoding.tokens())

In [16]:
print(encoding.word_ids())

[None, 0, 0, 0, 1, 1, 1, 1, 2, 2, 3, 3, 3, 4, 5, 5, 6, 6, 6, 6, 7, 7, 7, 7, 8, 8, 8, 9, 10, 10, 10, 11, 11, 11, 12, 12, 12, None]


In [17]:
# create dataset object

dataset = Dataset.from_list(examples)

eval_dataset = Dataset.from_list(eval_examples)

In [18]:
def tokenize_align_labels(example):
    tokens = example["tokens"]
    labels = example["labels"]
    
    encoding = tokenizer(
        tokens,
        is_split_into_words = True,
        truncation=True,
        max_length=512
    )

    word_ids = encoding.word_ids()

    aligned_labels = []

    previous_word_id = None

    for word_id in word_ids:
        if word_id is None:
           aligned_labels.append(-100)

        elif word_id != previous_word_id:
            aligned_labels.append(
                label2id[labels[word_id]]
            )
        else:
            aligned_labels.append(-100)


        previous_word_id = word_id

    encoding['labels'] = aligned_labels

    return encoding

### Strategi label subword (tambahan review)

Strategi yang dipakai adalah **first-subword labeling**:

- subword pertama dari setiap token asli menerima label BIO token tersebut;
- continuation subword, special token, dan padding menerima label `-100` agar diabaikan oleh loss;
- prediksi dikembalikan ke token asli dengan mengambil prediksi subword pertama;
- token kosong/whitespace ditolak karena tidak menghasilkan wordpiece yang dapat diberi label.

Fungsi tambahan di bawah mempertahankan perilaku fungsi Kevin, menambahkan validasi input dan fungsi untuk mengembalikan prediksi ke token asli. Nama `tokenize_align_labels` kemudian diarahkan ke versi tervalidasi sebelum cell `.map(...)` asli dijalankan.


In [19]:
IGNORE_INDEX = -100


def align_word_labels(word_ids, word_labels):
    """Berikan label hanya ke subword pertama setiap token asli."""
    aligned_labels = []
    seen_word_ids = set()

    for word_id in word_ids:
        if word_id is None:
            aligned_labels.append(IGNORE_INDEX)
            continue

        if not 0 <= word_id < len(word_labels):
            raise ValueError(
                f"word_id {word_id} di luar jumlah label {len(word_labels)}"
            )

        if word_id in seen_word_ids:
            aligned_labels.append(IGNORE_INDEX)
            continue

        label = word_labels[word_id]
        if label not in label2id:
            raise ValueError(f"Label BIO tidak dikenal: {label}")

        aligned_labels.append(label2id[label])
        seen_word_ids.add(word_id)

    return aligned_labels


def tokenize_and_align_labels(
    example,
    *,
    max_length=512,
    padding=False,
):
    """Tokenisasi token pre-split dan tambahkan label first-subword."""
    tokens = example["tokens"]
    labels = example["labels"]

    if len(tokens) != len(labels):
        raise ValueError("Jumlah token dan label berbeda")
    if any(not isinstance(token, str) or not token.strip() for token in tokens):
        raise ValueError("Token kosong/whitespace tidak diperbolehkan")

    encoding = tokenizer(
        tokens,
        is_split_into_words=True,
        truncation=True,
        max_length=max_length,
        padding=padding,
    )
    encoding["labels"] = align_word_labels(encoding.word_ids(), labels)
    return encoding


def predictions_to_word_labels(predicted_ids, word_ids, *, word_count):
    """Ambil prediksi subword pertama untuk setiap token asli tanpa shift."""
    if len(predicted_ids) != len(word_ids):
        raise ValueError("Jumlah prediksi dan word_id berbeda")

    word_predictions = [None] * word_count

    for prediction_id, word_id in zip(predicted_ids, word_ids):
        if word_id is None:
            continue
        if not 0 <= word_id < word_count:
            raise ValueError(f"word_id di luar rentang: {word_id}")
        if word_predictions[word_id] is not None:
            continue
        if prediction_id not in id2label:
            raise ValueError(f"ID prediksi tidak dikenal: {prediction_id}")

        word_predictions[word_id] = id2label[prediction_id]

    if any(label is None for label in word_predictions):
        raise ValueError(
            "Tidak semua token menerima prediksi; input mungkin terpotong"
        )

    return word_predictions


# Dipakai oleh cell dataset.map(...) milik Kevin yang berada setelah cell ini.
tokenize_align_labels = tokenize_and_align_labels


In [20]:
def assert_alignment_case(tokens, labels, **tokenizer_options):
    encoding = tokenize_and_align_labels(
        {"tokens": tokens, "labels": labels},
        **tokenizer_options,
    )
    word_ids = encoding.word_ids()
    aligned = encoding["labels"]

    assert len(word_ids) == len(aligned)
    for position, word_id in enumerate(word_ids):
        if word_id is None:
            # Special token dan padding harus diabaikan oleh loss.
            assert aligned[position] == IGNORE_INDEX

    for word_id, expected_label in enumerate(labels):
        positions = [
            position
            for position, current_word_id in enumerate(word_ids)
            if current_word_id == word_id
        ]
        assert positions, f"Token ke-{word_id} tidak menghasilkan subword"
        assert aligned[positions[0]] == label2id[expected_label]
        assert all(
            aligned[position] == IGNORE_INDEX
            for position in positions[1:]
        )

    predicted_ids = [
        label2id["O"] if label == IGNORE_INDEX else label
        for label in aligned
    ]
    restored = predictions_to_word_labels(
        predicted_ids,
        word_ids,
        word_count=len(tokens),
    )
    assert restored == labels
    return encoding


# Punctuation
assert_alignment_case(
    ["Jl", ".", "Merdeka", ","],
    ["B-JALAN", "I-JALAN", "I-JALAN", "O"],
)

# RT/RW
assert_alignment_case(
    ["RT", "03", "/", "RW", "05"],
    ["B-RT", "I-RT", "O", "B-RW", "I-RW"],
)

# Angka dan kode pos
assert_alignment_case(
    ["No", "12", "40123"],
    ["B-NOMOR", "I-NOMOR", "B-KODEPOS"],
)

# Special token dan padding mendapat -100.
assert_alignment_case(
    ["Bandung"],
    ["B-KOTA_KABUPATEN"],
    padding="max_length",
    max_length=12,
)

# Cari contoh yang benar-benar terpecah menjadi beberapa subword oleh mBERT.
subword_example = None
for candidate in [
    "ketidakbertanggungjawaban",
    "dipertanggungjawabkan",
    "mengadministrasikan",
]:
    candidate_encoding = tokenizer(
        [candidate],
        is_split_into_words=True,
    )
    if candidate_encoding.word_ids().count(0) > 1:
        subword_example = candidate
        break

assert subword_example is not None
assert_alignment_case([subword_example], ["B-DETAIL_LOKASI"])

# Empty input valid; hasilnya hanya special token yang berlabel -100.
empty_encoding = tokenize_and_align_labels({"tokens": [], "labels": []})
assert all(label == IGNORE_INDEX for label in empty_encoding["labels"])

# Whitespace-only token ditolak secara eksplisit.
try:
    tokenize_and_align_labels({"tokens": ["   "], "labels": ["O"]})
except ValueError:
    pass
else:
    raise AssertionError("Whitespace-only token seharusnya ditolak")

print("Semua self-check token alignment lulus.")


Semua self-check token alignment lulus.


In [21]:
# apply the whole preprocessing pipeline
tokenized_dataset = dataset.map(tokenize_align_labels)
eval_tokenized_dataset = eval_dataset.map(tokenize_align_labels)

Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:  14%|█▍        | 644/4500 [00:00<00:00, 6299.56 examples/s]

Map:  30%|███       | 1371/4500 [00:00<00:00, 5660.32 examples/s]

Map:  44%|████▍     | 2000/4500 [00:00<00:00, 3895.34 examples/s]

Map:  60%|█████▉    | 2692/4500 [00:00<00:00, 4723.06 examples/s]

...

Map:  90%|████████▉ | 4045/4500 [00:00<00:00, 5588.16 examples/s]

Map: 100%|██████████| 4500/4500 [00:00<00:00, 5292.96 examples/s]

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

Map:  98%|█████████▊| 737/750 [00:00<00:00, 7164.86 examples/s]

Map: 100%|██████████| 750/750 [00:00<00:00, 5989.05 examples/s]

In [22]:
tokenized_dataset

Dataset({
    features: ['id', 'categories', 'tokens', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 4500
})

# Compute Metrics

In [23]:
def compute_metrics(eval_preds):

    predictions, labels = eval_preds

    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for prediction, label in zip(predictions, labels):

        pred_labels = []
        true_label = []

        for pred, lab in zip(prediction, label):

            # Ignore special tokens and padding
            if lab == -100:
                continue

            pred_labels.append(id2label[pred])
            true_label.append(id2label[lab])

        true_predictions.append(pred_labels)
        true_labels.append(true_label)

    return {
        "accuracy": accuracy_score(
            true_labels,
            true_predictions
        ),
        "precision": precision_score(
            true_labels,
            true_predictions
        ),
        "recall": recall_score(
            true_labels,
            true_predictions
        ),
        "f1": f1_score(
            true_labels,
            true_predictions
        )
    }

# training

In [24]:
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-multilingual-cased",
    num_labels=len(data['label_order']),
    id2label=id2label,
    label2id=label2id
)

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Loading weights:   1%|          | 1/197 [00:00<?, ?it/s, Materializing param=bert.embeddings.LayerNorm.bias]

Loading weights:   1%|          | 1/197 [00:00<00:00, 1001.74it/s, Materializing param=bert.embeddings.LayerNorm.bias]

Loading weights:   1%|          | 2/197 [00:00<00:00, 666.98it/s, Materializing param=bert.embeddings.LayerNorm.weight]

Loading weights:   1%|          | 2/197 [00:00<00:00, 400.05it/s, Materializing param=bert.embeddings.LayerNorm.weight]

...

Loading weights:  99%|█████████▉| 196/197 [00:00<00:00, 1013.68it/s, Materializing param=bert.encoder.layer.11.output.dense.bias]

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1013.68it/s, Materializing param=bert.encoder.layer.11.output.dense.weight]

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1013.68it/s, Materializing param=bert.encoder.layer.11.output.dense.weight]

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1111.90it/s, Materializing param=bert.encoder.layer.11.output.dense.weight]


BertForTokenClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly in

In [25]:
training_args = TrainingArguments(
    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=5,

    weight_decay=0.01,

    logging_steps=50,
    report_to="none"
)

In [26]:
# Pastikan Trainer memakai seed eksplisit yang sama.
training_args.seed = SEED
training_args.data_seed = SEED

print(
    f"Trainer seed={training_args.seed}, "
    f"data_seed={training_args.data_seed}"
)


Trainer seed=42, data_seed=42


In [27]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    compute_metrics = compute_metrics,
    eval_dataset = eval_tokenized_dataset
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.017515,0.005210,0.999141,0.997444,0.997809,0.997626
2,0.006303,0.003538,0.999453,0.999269,0.998904,0.999087
3,0.002907,0.002826,0.999610,0.999087,0.999087,0.999087
4,0.001379,0.002032,0.999610,0.998722,0.998904,0.998813
5,0.001536,0.002223,0.999688,0.999452,0.999270,0.999361


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.93it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.92it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.97it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.80it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.80it/s]

TrainOutput(global_step=1410, training_loss=0.061076339819726155, metrics={'train_runtime': 3043.8975, 'train_samples_per_second': 7.392, 'train_steps_per_second': 0.463, 'total_flos': 716784579191592.0, 'train_loss': 0.061076339819726155, 'epoch': 5.0})

# test

In [28]:
with open('../../data/synthetic/test.json', 'r') as f:
    test_data = json.load(f)

test_examples = test_data['examples']
test_dataset = Dataset.from_list(test_examples)
test_tokenized_dataset = test_dataset.map(tokenize_align_labels)

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

Map:  17%|█▋        | 127/750 [00:00<00:00, 1175.24 examples/s]

Map:  96%|█████████▌| 721/750 [00:00<00:00, 3844.64 examples/s]

Map: 100%|██████████| 750/750 [00:00<00:00, 2477.64 examples/s]

In [29]:
trainer.evaluate(test_tokenized_dataset)

{'eval_loss': 0.00488318270072341,
 'eval_accuracy': 0.9994447088687927,
 'eval_precision': 0.9990685543964233,
 'eval_recall': 0.9988824734587446,
 'eval_f1': 0.9989755052621774,
 'eval_runtime': 22.2222,
 'eval_samples_per_second': 33.75,
 'eval_steps_per_second': 2.115,
 'epoch': 5.0}

# save model

In [30]:
# model.save_pretrained("/kaggle/working/bert_alamatin", from_pt=True) 